# Deploying Iris-detection model using Vertex AI


### Dataset

This tutorial uses R.A. Fisher's Iris dataset, a small and popular dataset for machine learning experiments. Each instance has four numerical features, which are different measurements of a flower, and a target label that
categorizes the flower into: **Iris setosa**, **Iris versicolour** and **Iris virginicadsdsadasdas**.

This tutorial uses [a version of the Iris dataset available in the
scikit-learn library](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_iris.html#sklearn.datasets.load_iris).

## Get started

### Install Vertex AI SDK for Python and other required packages



In [1]:

# Vertex SDK for Python
! pip3 install --upgrade --quiet  google-cloud-aiplatform joblib pandas scikit-learn dvc

### Set Google Cloud project information
Learn more about [setting up a project and a development environment](https://cloud.google.com/vertex-ai/docs/start/cloud-environment).

In [3]:
PROJECT_ID = "nifty-harmony-474217-q6"  # @param {type:"string"}
LOCATION = "us-central1"  # @param {type:"string"}

### Create a Cloud Storage bucket

Create a storage bucket to store intermediate artifacts such as datasets.

In [4]:
import pytest
import pandas as pd
import numpy as np
import joblib
from pathlib import Path


**If your bucket doesn't already exist**: Run the following cell to create your Cloud Storage bucket.

In [5]:
# Skipping bucket creation as it already exists
# ! gsutil mb -l {LOCATION} -p {PROJECT_ID} {BUCKET_URI}

### Initialize Vertex AI SDK for Python

To get started using Vertex AI, you must have an existing Google Cloud project and [enable the Vertex AI API](https://console.cloud.google.com/flows/enableapi?apiid=aiplatform.googleapis.com).

In [6]:
from google.cloud import aiplatform

aiplatform.init(project=PROJECT_ID, location=LOCATION, staging_bucket=BUCKET_URI)

### Import the required libraries

In [7]:
import os
import sys
from google.cloud import storage
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn import metrics
from sklearn.datasets import load_iris


### Configure resource names

Set a name for the following parameters:

`MODEL_ARTIFACT_DIR` - Folder directory path to your model artifacts within a Cloud Storage bucket, for example: "my-models/fraud-detection/trial-4"

`REPOSITORY` - Name of the Artifact Repository to create or use.

`IMAGE` - Name of the container image that is pushed to the repository.

`MODEL_DISPLAY_NAME` - Display name of Vertex AI model resource.

In [8]:
MODEL_ARTIFACT_DIR = "my-models/iris-classifier-week-1"  # @param {type:"string"}
REPOSITORY = "iris-classifier-repo"  # @param {type:"string"}
IMAGE = "iris-classifier-img"  # @param {type:"string"}
MODEL_DISPLAY_NAME = "iris-classifier"  # @param {type:"string"}

# Set the defaults if no names were specified
if MODEL_ARTIFACT_DIR == "[your-artifact-directory]":
    MODEL_ARTIFACT_DIR = "custom-container-prediction-model"

if REPOSITORY == "[your-repository-name]":
    REPOSITORY = "custom-container-prediction"

if IMAGE == "[your-image-name]":
    IMAGE = "sklearn-fastapi-server"

if MODEL_DISPLAY_NAME == "[your-model-display-name]":
    MODEL_DISPLAY_NAME = "sklearn-custom-container"

# Homework Pipeline

## Requirement 1: Importing data from the Google Storage Bucket

In [5]:
def load_iris_from_gc(bucket_name=TRAINING_DATA_BUCKET_NAME,blob_name=TRAINING_BLOB):
    client=storage.Client()
    bucket=client.bucket(bucket_name)
    blob=bucket.blob(blob_name)
    data_bytes=blob.download_as_bytes()
    df=pd.read_csv(pd.io.common.BytesIO(data_bytes))
    return df

iris_df=load_iris_from_gc()
data=iris_df
iris_df.head()

NameError: name 'TRAINING_DATA_BUCKET_NAME' is not defined

## Simple Decision Tree model
Build a Decision Tree model on iris data

In [10]:
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

mod_dt = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt.fit(X_train,y_train)


,criterion,'gini'
,splitter,'best'
,max_depth,3
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,1
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [11]:
prediction=mod_dt.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.983


In [12]:
os.makedirs('artifacts', exist_ok=True)
joblib.dump(mod_dt, "artifacts/model.joblib")

['artifacts/model.joblib']

### Storing output artifact in the artifact bucket 

In [13]:
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
!gsutil cp artifacts/model.joblib {BUCKET_URI}/{MODEL_ARTIFACT_DIR}/{timestamp}/model.joblib
print(f"{BUCKET_URI}/{MODEL_ARTIFACT_DIR}/{timestamp}/model.joblib")


Copying file://artifacts/model.joblib [Content-Type=application/octet-stream]...
/ [1 files][  2.5 KiB/  2.5 KiB]                                                
Operation completed over 1 objects/2.5 KiB.                                      
gs://mlops-course-nifty-harmony-474217-q6-unique/my-models/iris-classifier-week-1/20251012_114805/model.joblib


# Week 2: Implementing DVC and Git for Version Control

The goal is to use **DVC** for data and model artifacts and **Git** for code and DVC metafiles (`.dvc`).

## Setup: Initialize Git and DVC

In [14]:
GIT_INIT_STATUS = !git status
if 'not a git repository' in GIT_INIT_STATUS[0].lower():
    !git init
!dvc init
try:
    !dvc remote add -d -f myremote gs://{TRAINING_DATA_BUCKET_NAME}/dvc_store
except:
    !dvc remote add -d -f myremote gs://{TRAINING_DATA_BUCKET_NAME}/dvc_store


ERROR: failed to initiate DVC - '.dvc' exists. Use `-f` to force.
Setting 'myremote' as a default remote.


## Version 1: Initial Data and Model (150 rows)

In [15]:
# Save data locally to be tracked by DVC
iris_df.to_csv("iris.csv",index=False)
len(iris_df)

150

In [16]:
print("Adding data to DVC cache and creating metafiles...")
!dvc add artifacts/model.joblib iris.csv

print("Creating .gitignore files...")
!echo "iris.csv" > .gitignore
!echo "model.joblib" > artifacts/.gitignore

!git add Homework_Pipeline.ipynb .gitignore artifacts/.gitignore artifacts/model.joblib.dvc iris.csv.dvc
!git commit -m "Initial commit with DVC tracking for data and model (v1)"

!dvc push
print("push: Data and model artifacts pushed to DVC remote.")
MODEL_1_COMMIT = !git log -1 --pretty=format:"%H"
MODEL_1_COMMIT = MODEL_1_COMMIT[0]
print(f"Model 1 Commit ID: {MODEL_1_COMMIT}")

Adding data to DVC cache and creating metafiles...
 ⠋ Checking graph
  0% Adding...|             | artifacts/model.joblib |0/2 [00:00<?,     ?file/s]
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Checking out /home/jupyter/artifacts/m0/1 [00:00<?,    ?files/s]
  0% Adding...|                           | iris.csv |0/2 [00:00<?,     ?file/s]
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Checking out /home/jupyter/iris.csv   0/1 [00:00<?,    ?files/s]
100% Adding...|████████████████████████████████████████|2/2 [00:00, 34.11file/s]

To track the changes with g

## Version 2: Augmented Data and Retrained Model (160 rows)

In [17]:
df=load_iris(as_frame=True) # Load the original data source from sklearn
new_rows=df.frame.sample(10,replace=True)
new_rows['target'] = new_rows['target'].map({i: name for i, name in enumerate(df.target_names)})
new_rows=new_rows.rename(columns={'sepal length (cm)':'sepal_length',
                                  'sepal width (cm)':'sepal_width',
                                 'petal length (cm)':'petal_length',
                                 'petal width (cm)':'petal_width',
                                 'target':'species'})
iris_df_v2=pd.concat([iris_df,new_rows])

print(f"New dataset size: {len(iris_df_v2)}")
iris_df_v2.tail(5)

New dataset size: 160


,sepal_length,sepal_width,petal_length,petal_width,species
37,4.9,3.6,1.4,0.1,setosa
22,4.6,3.6,1.0,0.2,setosa
125,7.2,3.2,6.0,1.8,virginica
59,5.2,2.7,3.9,1.4,versicolor
116,6.5,3.0,5.5,1.8,virginica


In [18]:
data=iris_df_v2
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_train = train[['sepal_length','sepal_width','petal_length','petal_width']]
y_train = train.species
X_test = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test = test.species

mod_dt_v2 = DecisionTreeClassifier(max_depth = 3, random_state = 1)
mod_dt_v2.fit(X_train,y_train)


,criterion,'gini'
,splitter,'best'
,max_depth,3
,min_samples_split,2
,min_samples_leaf,1
,min_weight_fraction_leaf,0.0
,max_features,None
,random_state,1
,max_leaf_nodes,None
,min_impurity_decrease,0.0
,class_weight,None


In [19]:
prediction=mod_dt_v2.predict(X_test)
print('The accuracy of the Decision Tree is',"{:.3f}".format(metrics.accuracy_score(prediction,y_test)))

The accuracy of the Decision Tree is 0.891


In [20]:
print("Saving V2 data and model locally...")
iris_df_v2.to_csv("iris.csv",index=False)
joblib.dump(mod_dt_v2, "artifacts/model.joblib")

Saving V2 data and model locally...


['artifacts/model.joblib']

In [21]:
print("Adding artifacts/model.joblib, iris.csv to DVC cache...")
!dvc add artifacts/model.joblib iris.csv

!git add iris.csv.dvc artifacts/model.joblib.dvc Homework_Pipeline.ipynb
!git commit -m "Retrained model v2 on augmented data (160 rows)"

!dvc push
print("push: Data and model artifacts pushed to DVC remote.")
MODEL_2_COMMIT = !git log -1 --pretty=format:"%H"
MODEL_2_COMMIT = MODEL_2_COMMIT[0]
print(f"Model 2 Commit ID: {MODEL_2_COMMIT}")

Adding artifacts/model.joblib, iris.csv to DVC cache...
 ⠋ Checking graph
  0% Adding...|             | artifacts/model.joblib |0/2 [00:00<?,     ?file/s]
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Adding artifacts/model.joblib to cache0/1 [00:00<?,     ?file/s]
                                                                                
!
  0%|          |Checking out /home/jupyter/artifacts/m0/1 [00:00<?,    ?files/s]
  0% Adding...|                           | iris.csv |0/2 [00:00<?,     ?file/s]
!
                                                                                
!
  0% Checking cache in '/home/jupyter/.dvc/cache/files/md5'| |0/? [00:00<?,    ?
                                                                                
!
  0%|          |Addin

## Verification: Switch Between Model and Data Versions

In [22]:
print(f"Checking out Git commit: {MODEL_1_COMMIT}")
!git checkout {MODEL_1_COMMIT}
print("Switched to commit 8ebe656")

print("Restoring DVC-tracked files...")
!dvc checkout
print("DVC checkout successful. Data and model files should be V1.")

iris_df_v1_restored = pd.read_csv("iris.csv")
print(f"Local iris.csv row count: {len(iris_df_v1_restored)}")

mod_dt_v1_restored = joblib.load("artifacts/model.joblib")

data=iris_df_v1_restored
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_test_v1 = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test_v1 = test.species

prediction_v1=mod_dt_v1_restored.predict(X_test_v1)
print('Accuracy of Restored Model V1 is',"{:.3f}".format(metrics.accuracy_score(prediction_v1,y_test_v1)))

Checking out Git commit: 12e5faf83bc085e1b5ddc281f7fc3121200665d7
D	SDK_Custom_Container_Prediction.ipynb
any of your branches:

  4a3637b Retrained model v2 on augmented data (160 rows)

If you want to keep it by creating a new branch, this may be a good time
to do so with:

 git branch <new-branch-name> 4a3637b

HEAD is now at 12e5faf Initial commit with DVC tracking for data and model (v1)
Switched to commit 8ebe656
Restoring DVC-tracked files...
Building workspace index                              |3.00 [00:00, 11.3entry/s]
Comparing indexes                                    |4.00 [00:00, 1.29kentry/s]
Applying changes                                      |2.00 [00:00,   415file/s]
M       artifacts/model.joblib
M       iris.csv
DVC checkout successful. Data and model files should be V1.
Local iris.csv row count: 150
Accuracy of Restored Model V1 is 0.983


In [23]:
print(f"Checking out Git commit: {MODEL_2_COMMIT}")
!git checkout master
print("Switched to branch 'master'")

print("Restoring DVC-tracked files...")
!dvc checkout
print("DVC checkout successful. Data and model files should be V2.")

iris_df_v2_restored = pd.read_csv("iris.csv")
print(f"Local iris.csv row count: {len(iris_df_v2_restored)}")

mod_dt_v2_restored = joblib.load("artifacts/model.joblib")

data=iris_df_v2_restored
train, test = train_test_split(data, test_size = 0.4, stratify = data['species'], random_state = 42)
X_test_v2 = test[['sepal_length','sepal_width','petal_length','petal_width']]
y_test_v2 = test.species

prediction_v2=mod_dt_v2_restored.predict(X_test_v2)
print('Accuracy of Restored Model V2 is',"{:.3f}".format(metrics.accuracy_score(prediction_v2,y_test_v2)))

Checking out Git commit: 4a3637b1204a26125fea1d2d04ef94f5fb3af36c
D	SDK_Custom_Container_Prediction.ipynb
any of your branches:

  12e5faf Initial commit with DVC tracking for data and model (v1)
  d682768 Pipeline removed

If you want to keep them by creating a new branch, this may be a good time
to do so with:

 git branch <new-branch-name> 12e5faf

Switched to branch 'master'
Switched to branch 'master'
Restoring DVC-tracked files...
Building workspace index                              |3.00 [00:00, 11.6entry/s]
Comparing indexes                                     |4.00 [00:00,  968entry/s]
Applying changes                                      |2.00 [00:00,   378file/s]
M       artifacts/model.joblib
M       iris.csv
DVC checkout successful. Data and model files should be V2.
Local iris.csv row count: 160
Accuracy of Restored Model V2 is 0.891
